# AEGIS-SQL × Qwen2.5-Coder 1.5B

LitE-SQL-inspired pretrained baseline. **Runtime → Change runtime type → GPU**를 먼저 선택하세요.

이 노트북은 base Qwen과 AEGIS flywheel QLoRA 이후 모델을 **같은 KorFin evaluator**로 비교합니다. 성능 수치는 실제 실행 결과가 생긴 뒤에만 사용합니다.

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime을 선택한 뒤 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. 저장소 준비
이미 clone한 세션이라도 재현성을 위해 main을 새로 받습니다.

In [ ]:
!rm -rf /content/aegis-sql
!git clone -q https://github.com/sokldjs554/aegis-sql.git /content/aegis-sql
%cd /content/aegis-sql
!git rev-parse HEAD

## 2. 먼저 smoke run
64 train / 32 dev / 10 eval item으로 wiring과 GPU stack을 확인합니다. 이 결과는 포트폴리오 성능으로 쓰지 않습니다.

In [ ]:
!SMOKE=1 bash scripts/run_qwen_colab.sh

## 3. Full run
Smoke가 성공한 뒤 실행합니다. 기존 skeleton-cluster flywheel split을 그대로 사용하고, base → QLoRA → adapted 순서로 전체 answerable KorFin을 평가합니다.

In [ ]:
!bash scripts/run_qwen_colab.sh

## 4. 결과 확인
`items`가 전체 평가인지 반드시 확인하고, base/adapted EX와 difficulty별 EX, latency, peak CUDA memory를 함께 기록합니다.

In [ ]:
import json
from pathlib import Path
for p in [Path('reports/qwen2.5-coder-1.5b-base.json'), Path('reports/qwen2.5-coder-1.5b-adapted.json')]:
    d = json.loads(p.read_text())
    print('\n', p.name)
    print('items =', d['items'])
    print('EX =', f"{d['execution_accuracy']:.1%}")
    print('difficulty =', {k: f"{v['ex']:.1%}" for k, v in d['by_difficulty'].items()})
    print('latency_ms =', d['latency_ms'])
    if 'max_cuda_memory_bytes' in d:
        print('peak CUDA GiB =', round(d['max_cuda_memory_bytes'] / 1024**3, 2))
print('\nmanifest:')
print(Path('data/generated/hf/qwen2.5-coder-1.5b/experiment_manifest.json').read_text())

## 다음 단계
1.5B full 결과를 먼저 고정한 뒤에만 3B를 같은 데이터·LoRA 설정으로 반복합니다. GPU 자원 때문에 batch/max-length 등을 바꾸면 동일 실험으로 섞지 말고 별도 run으로 기록합니다.